# Limburg map to Heerlen edge table csv

In [1]:
# Build edge-level connectivity + travel time tables from the routing graph
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox

In [2]:
# Build a drivable graph for Heerlen + 1 km in every direction
heerlen_boundary = ox.geocode_to_gdf("Heerlen, Limburg, Netherlands")

# Buffer in a metric CRS so 1 km is accurate
heerlen_boundary_m = heerlen_boundary.to_crs(epsg=3857)
expanded_geom_m = heerlen_boundary_m.geometry.iloc[0].buffer(250)
expanded_geom = gpd.GeoSeries([expanded_geom_m], crs="EPSG:3857").to_crs(epsg=4326).iloc[0]

route_graph = ox.graph_from_polygon(expanded_geom, network_type="drive")
print("Loaded graph for Heerlen + 250 m buffer.")

Loaded graph for Heerlen + 250 m buffer.


In [3]:
# 1) Add speed and travel time attributes (travel_time in seconds)
route_graph_tt = ox.add_edge_speeds(route_graph)
route_graph_tt = ox.add_edge_travel_times(route_graph_tt)

# 2) Convert graph edges to a table
edges_gdf = ox.graph_to_gdfs(route_graph_tt, nodes=False, edges=True).reset_index()

# Keep common attributes if present
cols = ["u", "v", "key", "name", "highway", "length", "speed_kph", "travel_time", "geometry"]
edge_table = edges_gdf[[c for c in cols if c in edges_gdf.columns]].copy()

# Clean name/highway fields (they can be lists)
def _to_text(value):
    if isinstance(value, list):
        return " | ".join(map(str, value))
    return value

if "name" in edge_table.columns:
    edge_table["name"] = edge_table["name"].apply(_to_text)
if "highway" in edge_table.columns:
    edge_table["highway"] = edge_table["highway"].apply(_to_text)

# Add human-readable travel time
if "travel_time" in edge_table.columns:
    edge_table["travel_time_min"] = edge_table["travel_time"] / 60.0

print(f"Directed road segments (edges): {len(edge_table):,}")
edge_table.head(10)

# Export expanded-area edge table
edge_table.to_csv("../output/heerlen_edge_table.csv", index=False)
print("Saved: ../output/heerlen_edge_table.csv")

Directed road segments (edges): 7,183
Saved: ../output/heerlen_edge_table.csv


In [4]:
# Build per-transport edge table with mode-specific speeds and travel times
import ast

WALK_SPEED_KPH = 5.0
BIKE_SPEED_KPH = 15.0
DEFAULT_CAR_SPEED_KPH = 30.0

# Reuse edge_table from earlier cell if available, otherwise load the exported table
if "edge_table" in globals() and isinstance(edge_table, pd.DataFrame):
    base_edges = edge_table.copy()
else:
    base_edges = pd.read_csv("../output/heerlen_edge_table.csv")


def parse_highway_tags(value):
    if pd.isna(value):
        return set()

    if isinstance(value, (list, tuple, set)):
        return {str(v).strip().lower() for v in value if str(v).strip()}

    text = str(value).strip()
    if not text:
        return set()

    # Handle list-like strings such as "['primary', 'service']"
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple, set)):
                return {str(v).strip().lower() for v in parsed if str(v).strip()}
        except (SyntaxError, ValueError):
            pass

    # Handle the format produced earlier: "a | b"
    if "|" in text:
        return {part.strip().lower() for part in text.split("|") if part.strip()}

    return {text.lower()}


# Accessibility sets
foot_accessible_tags = {
    "footway", "pedestrian", "path", "steps", "sidewalk", "corridor", "living_street",
    "residential", "service", "unclassified", "road", "track", "cycleway",
    "tertiary", "tertiary_link", "secondary", "secondary_link", "primary", "primary_link"
}

bike_accessible_tags = {
    "cycleway", "cycleway_link", "path", "track", "living_street", "service", "road",
    "residential", "unclassified", "tertiary", "tertiary_link", "secondary",
    "secondary_link", "primary", "primary_link"
}

car_accessible_tags = {
    "motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link",
    "secondary", "secondary_link", "tertiary", "tertiary_link", "residential",
    "unclassified", "service", "road", "living_street", "track"
}


def travel_time_seconds(length_m, speed_kph):
    meters_per_sec = (speed_kph * 1000.0) / 3600.0
    return length_m / meters_per_sec if meters_per_sec > 0 else np.nan


records = []
for _, row in base_edges.iterrows():
    tags = parse_highway_tags(row.get("highway", np.nan))
    length_m = float(row.get("length", np.nan)) if pd.notna(row.get("length", np.nan)) else np.nan

    if not tags or pd.isna(length_m):
        continue

    row_dict = row.to_dict()

    if tags & foot_accessible_tags:
        rec = row_dict.copy()
        rec["transportation_type"] = "pedestrian"
        rec["speed_kph"] = WALK_SPEED_KPH
        rec["travel_time"] = travel_time_seconds(length_m, WALK_SPEED_KPH)
        rec["travel_time_min"] = rec["travel_time"] / 60.0
        records.append(rec)

    if tags & bike_accessible_tags:
        rec = row_dict.copy()
        rec["transportation_type"] = "bike"
        rec["speed_kph"] = BIKE_SPEED_KPH
        rec["travel_time"] = travel_time_seconds(length_m, BIKE_SPEED_KPH)
        rec["travel_time_min"] = rec["travel_time"] / 60.0
        records.append(rec)

    if tags & car_accessible_tags:
        rec = row_dict.copy()
        car_speed = rec.get("speed_kph", np.nan)
        car_speed = float(car_speed) if pd.notna(car_speed) else DEFAULT_CAR_SPEED_KPH
        if car_speed <= 0:
            car_speed = DEFAULT_CAR_SPEED_KPH
        rec["transportation_type"] = "car"
        rec["speed_kph"] = car_speed
        rec["travel_time"] = travel_time_seconds(length_m, car_speed)
        rec["travel_time_min"] = rec["travel_time"] / 60.0
        records.append(rec)


edge_table_traveltypes = pd.DataFrame(records)

# Keep a readable column order if those columns exist
preferred_cols = [
    "u", "v", "key", "name", "highway", "transportation_type", "length",
    "speed_kph", "travel_time", "travel_time_min", "geometry"
]
existing_preferred = [c for c in preferred_cols if c in edge_table_traveltypes.columns]
remaining_cols = [c for c in edge_table_traveltypes.columns if c not in existing_preferred]
edge_table_traveltypes = edge_table_traveltypes[existing_preferred + remaining_cols]

output_path = "../output/heerlen_edge_table_traveltypes.csv"
edge_table_traveltypes.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Total records: {len(edge_table_traveltypes):,}")
print("Records per transportation type:")
print(edge_table_traveltypes["transportation_type"].value_counts())

edge_table_traveltypes.head(10)

Saved: ../output/heerlen_edge_table_traveltypes.csv
Total records: 21,096
Records per transportation type:
transportation_type
car           7142
pedestrian    6977
bike          6977
Name: count, dtype: int64


,u,v,key,name,highway,transportation_type,length,speed_kph,travel_time,travel_time_min,geometry
0,41941190,41940361,0,NaN,motorway,car,184.415130,40.0,16.597362,0.276623,"LINESTRING (6.021233 50.821986, 6.0221654 50.8..."
1,41941240,712758381,0,NaN,motorway_link,car,37.950937,100.0,1.366234,0.022771,"LINESTRING (6.0223386 50.8214182, 6.022286 50...."
2,41941852,41941190,0,NaN,motorway_link,car,46.927499,60.0,2.815650,0.046927,"LINESTRING (6.0207387 50.8222699, 6.0208651 50..."
3,41942044,41944458,0,NaN,motorway,car,300.546619,100.0,10.819678,0.180328,"LINESTRING (6.0210875 50.8224672, 6.0187792 50..."
4,41942044,41942494,0,NaN,motorway_link,car,58.733325,100.0,2.114400,0.035240,"LINESTRING (6.0210875 50.8224672, 6.0209906 50..."
5,41943918,41944458,0,NaN,motorway_link,car,86.537038,100.0,3.115333,0.051922,"LINESTRING (6.0198126 50.8243229, 6.0196696 50..."
6,41944341,41941190,0,NaN,motorway,car,355.484137,60.0,21.329048,0.355484,"LINESTRING (6.0184986 50.8246761, 6.0198755 50..."
7,41944341,13136760726,0,NaN,motorway_link,car,43.250817,70.0,2.224328,0.037072,"LINESTRING (6.0184986 50.8246761, 6.0185763 50..."
8,41944458,41948483,0,NaN,motorway,car,502.798486,100.0,18.100745,0.301679,"LINESTRING (6.0187792 50.824743, 6.0186237 50...."
9,41948483,41961522,0,NaN,motorway,car,1631.606731,100.0,58.737842,0.978964,"LINESTRING (6.0149201 50.8285514, 6.0115205 50..."
